# DOI / PMID / 标题精确解析

> 快速拉取元数据、全文和引用证据，科研 Agent 的基础入口

**场景**: 用户已有 DOI、PMID 或论文标题，只想快速拉取元数据、全文和引用证据。这是科研 Agent 最基础的入口操作。

**使用接口**: meta-search, content

**预估调用量**: ~3–5 次 API 调用 / 一次解析

---


## Step 1: 环境准备

安装依赖并配置 API Token


In [ ]:
!pip install httpx
import os
os.environ["SCIVERSE_API_TOKEN"] = "sv-your-token-here"  # 替换为你的真实值


## Step 2: 通过 DOI 精确查找文献

用 meta-search 按 DOI 精确匹配


In [ ]:
import os
import asyncio
import httpx

BASE = "https://api.sciverse.space"
TOKEN = os.environ["SCIVERSE_API_TOKEN"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def resolve_doi(doi: str):
    """\u901a\u8fc7 DOI \u7cbe\u786e\u67e5\u627e\u6587\u732e\u5143\u6570\u636e"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/meta-search", headers=HEADERS,
            json={
                "query": doi,
                "filters": [{"field": "doi", "operator": "FILTER_OP_EQ", "value": doi}],
                "page": 1, "page_size": 1
            }
        )
        resp.raise_for_status()
        data = resp.json()
        if data["total_count"] > 0:
            return data["results"][0]
        return None

async def resolve_title(title: str):
    """\u901a\u8fc7\u6807\u9898\u6a21\u7cca\u67e5\u627e"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{BASE}/meta-search", headers=HEADERS,
            json={"query": title, "filters": [], "page": 1, "page_size": 5}
        )
        resp.raise_for_status()
        return resp.json()["results"]

# \u793a\u4f8b\uff1a\u901a\u8fc7 DOI \u89e3\u6790
paper = asyncio.run(resolve_doi("10.1038/s41586-021-03819-2"))
if paper:
    print(f"Title: {paper['title']}")
    print(f"Venue: {paper.get('publication_venue_name', 'N/A')}")
    print(f"Year: {paper.get('publication_published_year', 'N/A')}")
    print(f"Doc ID: {paper.get('doc_id', 'N/A')}")


## Step 3: 读取全文摘要

用 content 接口拉取论文开头段落


In [ ]:
async def read_abstract(doc_id: str):
    """\u8bfb\u53d6\u8bba\u6587\u5f00\u5934 1500 \u5b57\u7b26\u4f5c\u4e3a\u6458\u8981"""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.get(
            f"{BASE}/content", headers=HEADERS,
            params={"doc_id": doc_id, "offset": 0, "limit": 1500}
        )
        resp.raise_for_status()
        return resp.json()["text"]

if paper and paper.get("doc_id"):
    abstract = asyncio.run(read_abstract(paper["doc_id"]))
    print(f"\
Full text preview:\
{abstract[:500]}...")


## 注意事项

- DOI 精确查询使用 FILTER_OP_EQ 操作符
- 如果 DOI 查不到，可回退到标题模糊查询
- content 接口返回的是 text 字段，非 content
- 这是科研 Agent 最基础的入口操作，建议封装为通用工具函数


---

## 下一步

- [申请 API Token](https://sciverse.opendatalab.com/docs#auth)
- [查看完整 API 文档](https://sciverse.opendatalab.com/docs#sciverse/api)
- [更多 Cookbook 案例](https://sciverse.opendatalab.com/docs#cookbook)
